In [1]:
#from darksun.show_trials import plot#, image_plot
#
#plot(dmapx)
#plot(dmaps=(dmapx, dmapy), ncols=2)
#image_plot(
#    smoothed,
#    title='Smoothed Detector',
#    cbarlabel='counts',
#    cmap='hot',
#)

In [4]:
from typing import Any
from numpy.typing import NDArray


def unframe(
    data: NDArray,
    unframe_y: int | tuple[int | None, int | None] | None = None,
    unframe_x: int | tuple[int | None, int | None] | None = None,
) -> NDArray:
    """
    Unframes a 2D array by returning a `sliced view` of the array
    along the specified axis.

    Args:
        data (NDArray):
            Input 2D array.
        unframe_y (int | tuple[int | None, int | None] | None, optional (default=`None`)):
            Unframing factor over the y axis (rows).
            * If `int`, crops symetrically: `data[y:-y, :]`;
            * If `tuple(start, stop)`, crops asymetrically: `data[start : stop, :]`;
            * `None` in the tuple means no crop on that edge.
        unframe_x (int | tuple[int | None, int | None] | None, optional (default=`None`)):
            Unframing factor over the x axis (columns), same behavior as `unframe_y`.

    Returns:
        output (NDArray): View of the unframed array.
    
    Examples:
        >>> a = np.ones((10, 10))
        ...
        >>> b = unframe(a)              # same as 'a', shape: (10, 10)
        >>> c = unframe(a, 2)           # same as a[2:-2, :], shape: (6, 10)
        >>> d = unframe(a, 2, 3)        # same as a[2:-2, 3:-3], shape: (6, 4)
        >>> e = unframe(a, (2, None))   # same as a[2:, :], shape: (8, 10)
        >>> f = unframe(a, (None, -2))  # same as a[:-2, :], shape: (8, 10)
    """
    def is_valid(*factors) -> bool:
        """Checks if unframe factors are not floats."""
        return not isinstance(*factors, float)

    def setup_slice(factor: Any) -> slice:
        """Slice object config for array indexes."""
        if factor is None:
            i, j = (None, None)
        elif isinstance(factor, int):
            i, j = (factor, -factor if factor != 0 else None)
        else:
            i, j = factor
        return slice(i, j)
    
    if not (is_valid(unframe_y) or is_valid(unframe_x)):
        raise ValueError("Unframe factors must be positive integers.")
    
    unframe_f = (unframe_y, unframe_x)
    if not any(unframe_f):
        return data

    return data[*tuple(map(setup_slice, unframe_f))]

In [5]:
import numpy as np
from darksun.images import upscale

a = np.ones((10, 10))

b = unframe(a)              # same as 'a', shape: (10, 10)
c = unframe(a, 2.0)           # same as a[2:-2, :], shape: (6, 10)
d = unframe(a, 2, 3)        # same as a[2:-2, 3:-3], shape: (6, 4)
e = unframe(a, (2, None))   # same as a[2:, :], shape: (8, 10)
f = unframe(a, (None, -2))  # same as a[:-2, :], shape: (8, 10)

print(tuple(arr.shape for arr in (a, b, c, d, e, f)))

TypeError: cannot unpack non-iterable float object